# Toric code — exercise

**Goals**

- Create $\mathbb Z_2$-symmetric tensors with YASTN.
- Build a toric-code PEPS from a local parity constraint.
- Verify its stabilizers and extract the topological correction to its Rényi entropy.

**Before you start**

- Select a Python kernel with YASTN installed; see the [installation instructions](../README.md#installation).

- Run cells from top to bottom. Replace each `raise NotImplementedError(...)`
  with your implementation, then rerun that cell and its checks.
  Only the tensor construction and contractions are exercises; CTMRG, plotting,
  and index bookkeeping are provided.


In [ ]:
# Provided: imports and symmetry configuration.
from itertools import product

import numpy as np
import matplotlib.pyplot as plt
import yastn

# Use the NumPy backend and Z2 symmetry throughout.
z2_config = yastn.make_config(backend="np", sym="Z2", default_dtype="float64")


## 1. Create $\mathbb Z_2$-symmetric tensors

- A tensor leg is an index. YASTN groups the states on each leg into **charge sectors**
  and stores only blocks that obey charge conservation.
  Start with a rank-two tensor (a matrix) $M$. Each YASTN `Leg` specifies:

    - `s`: the signature, $+1$ (ingoing) or $-1$ (outgoing).
    - `t`: the charge labels, here `(0, 1)` for even and odd parity.
    - `D`: the dimension of each sector, here `(2, 1)`.

- **Arrow convention:** arrows point into a tensor for signature $+1$ and away
  from it for $-1$. Contracted legs have opposite signatures.
  Numbers in parentheses, such as `(0)`, label zero-based leg positions.

- For example, the matrix below has leg order `(q1, q2)` and signatures `(+,-)`:

  ```text
  q1(0) ---> [M] ---> q2(1)
  ```

- Each leg has dimension $2+1=3$. `leg.conj()` reverses its signature.
  The tensor's total charge is `n`. Here charge conservation means

  $$
  (q_1-q_2)\bmod2=n.
  $$

- For `n=0`, only equal-charge blocks are allowed:

  $$
  M=\begin{pmatrix}M_{00}&0\\0&M_{11}\end{pmatrix},
  $$

  where $M_{00}$ is a $2\times2$ block and $M_{11}$ is a $1\times1$ block.
  The off-diagonal blocks are symmetry-forbidden and are not stored.

- For $\mathbb Z_2$, charges add modulo two: $1+1=0$ and $-q=q$.
  The arrows still specify which legs can be contracted.
  Run the worked example below: `yastn.zeros` allocates allowed blocks and
  `set_block` fills them.


In [ ]:
# Provided: worked example of symmetric blocks.
# Two even states and one odd state on each leg.
leg = yastn.Leg(z2_config, s=1, t=(0, 1), D=(2, 1))
M = yastn.zeros(config=z2_config, legs=(leg, leg.conj()), n=0)

M.set_block(ts=(0, 0), val=np.array([[1., 2.], [3., 4.]]))
M.set_block(ts=(1, 1), val=np.array([[5.]]))

print("Dense shape:", M.get_shape())
print("Stored blocks (q1, q2):", M.get_blocks_charge())
print("Dense representation (missing blocks become zeros):")
print(M.to_numpy())


### 1.1. Two tensor operations used below

- `yastn.einsum` follows the usual index notation. For example,

  ```python
  C = yastn.einsum("ij,jk->ik", M, M)
  ```

  Here `j` is summed, and the labels after `->` set the output leg order.
  Contracted legs must have opposite signatures; `M.conj()` conjugates the entries
  and reverses all signatures.

- **Fusion** combines several indices into one. For a five-leg tensor `T`,
  `T.fuse_legs(axes=(0,(1,2),(3,4)), mode="hard")` returns three legs:
  `[0, (1,2), (3,4)]`. We use this to group ket and bra spin indices in section 5.
  The fusion calls are provided.


## 2. Toric-code ground state

- The toric code places one qubit on each edge of a square lattice.
  With Pauli matrices $X$ and $Z$, its Hamiltonian is

  $$
  H=-\sum_v A_v-\sum_p B_p,\qquad
  A_v=\prod_{e\ni v}Z_e,\qquad
  B_p=\prod_{e\in\partial p}X_e.
  $$

- Here $v$ is a vertex, $p$ a plaquette, and each product acts on four edges.
  These operators commute. Ground states satisfy $A_v=B_p=+1$ everywhere.

### 2.1. The closed-loop wavefunction

- Work in the $Z$ basis: $Z_e|s_e\rangle=(-1)^{s_e}|s_e\rangle$.
  Draw an occupied edge when $s_e=1$.

- $A_v=+1$ requires an even number of occupied edges at each vertex, so no loop can end there.

- $B_p$ flips the four bits around a plaquette, preserving this even-parity condition.

- On a finite lattice, the equal-amplitude loop state is

  $$
  |\psi_{\mathrm{loop}}\rangle
  =\frac{1}{\sqrt{\mathcal N}}\sum_{C\in\mathcal C}|C\rangle,
  $$

  where $\mathcal C$ contains all even-parity configurations and $\mathcal N=|\mathcal C|$.
  Plaquette flips permute these configurations, so this state has $A_v=B_p=+1$.
  We will represent it with local tensors and repeat them on the infinite lattice.


### 2.2. Encode the four-edge constraint with $\mathbb Z_2$ symmetry

- The even-parity constraint at each vertex can be encoded in a four-leg tensor $Q$.
  We can write $Q$ as a **$\mathbb Z_2$-symmetric tensor** by identifying each edge bit
  $s_e$ with a charge $q_e=s_e$, with one state in each sector $q_e=0,1$.

- Choose signatures `(+,+,-,-)` in `(u,l,d,r)` order and total charge `n=0`.
  Charge conservation then requires

  $$
  (q_u+q_l-q_d-q_r)\bmod2=0.
  $$

- Since $-q=q$ modulo two, this condition says that the four edge bits have even parity.
  Set every allowed block to one to obtain

  $$
  Q_{s_u s_l s_d s_r}=
  \begin{cases}
  1,&s_u+s_l+s_d+s_r=0\pmod2,\\
  0,&\text{otherwise}.
  \end{cases}
  $$

  ```text
                                            s_u(0)
                                              |
                                              v
                                  s_l(1) ---> [Q] ---> s_r(3)
                                              |
                                              v
                                            s_d(2)
  ```

- Place $Q$ at every original vertex. The amplitude of a fixed spin configuration is

  $$
  \Psi(\{s_e\})\propto\prod_v Q_{s_u(v)s_l(v)s_d(v)s_r(v)}.
  $$

  This gives equal weight to every closed-loop configuration and zero to the others.
  Use `yastn.ones` to fill the eight symmetry-allowed blocks with one.

**Exercise 2:** construct `Q` using the supplied leg, signatures `(+,+,-,-)`,
and `n=0`. Fill all allowed blocks with one.


In [ ]:
def constraint_tensor(config):
    """Four-edge even-parity constraint at an original lattice vertex."""
    leg = yastn.Leg(config, s=1, t=(0, 1), D=(1, 1))
    # TODO: Construct the neutral four-leg tensor, with leg order (u, l, d, r).
    raise NotImplementedError("Exercise 2: construct Q.")


Q_z2 = constraint_tensor(z2_config)
print(f"Z2: {len(Q_z2.get_blocks_charge())} stored scalars / 16 dense entries")
print("Allowed (s_u, s_l, s_d, s_r):", Q_z2.get_blocks_charge())
assert len(Q_z2.get_blocks_charge()) == 8
assert Q_z2.s == (1, 1, -1, -1)


### 2.3. Visualize the allowed vertices

- The provided helper reads the charge labels from `Q_z2` to draw its eight allowed vertices.

- A thick segment denotes an occupied edge; a faint segment denotes an empty edge.

- Check that every vertex has even occupancy. Arrows indicate leg signatures.


In [ ]:
# Provided: vertex visualization.
def plot_vertices(Q):
    """Draw allowed charges; arrows point into + legs and away from - legs."""
    vertices = list(Q.get_blocks_charge())
    fig, axes = plt.subplots(2, 4, figsize=(8, 3.6))
    endpoints = [(0, 1), (-1, 0), (0, -1), (1, 0)]  # up, left, down, right
    for ax, charges in zip(axes.flat, vertices):
        for q, signature, (x, y) in zip(charges, Q.s, endpoints):
            color = "black" if q else "0.75"
            ax.plot([0, x], [0, y], color=color, linewidth=3 if q else 1)
            start, end = (0.75, 0.4) if signature == 1 else (0.4, 0.75)
            ax.annotate("", xy=(end*x, end*y), xytext=(start*x, start*y),
                        arrowprops={"arrowstyle": "->", "color": color,
                                    "lw": 1.2, "mutation_scale": 9})
        ax.plot(0, 0, "ko", markersize=4)
        ax.set_title(str(charges), fontsize=10)
        ax.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), aspect="equal")
        ax.axis("off")
    for ax in axes.flat[len(vertices):]:
        ax.axis("off")
    fig.suptitle("Z2: even parity - (u, l, d, r)")
    fig.tight_layout()
    return fig


plot_vertices(Q_z2)
plt.show()


### 2.4. Expose the physical spins with $P$

- Summing every edge index of the $Q$ network counts closed-loop configurations.
  A wavefunction must retain one open index for each physical spin.
  Insert a **copy tensor** on every edge:

  $$
  P^s_{q,q'}=\delta_{q,q'}\delta_{s,q}.
  $$

- It copies the shared virtual bit onto the physical spin: $s=q=q'$.
  The network now has $Q$ at every vertex and $P$ on every edge:

  <p align="center">
    <img src="figures/toric_code_edge_arrows.png" alt="Original square lattice with Q at vertices, P on edges, and an orange physical spin leg on each P" width="850">
  </p>

- In code, `P[s,q,q_next]` has signatures `(+,+,-)`.
  Its two virtual legs carry the same charge $q=0,1$.
  The physical leg is a single charge-zero sector of dimension two, with basis $|0\rangle,|1\rangle$.

- The copy tensor is provided; run the next cell.


In [ ]:
# Provided: physical copy tensor.
def physical_copy(config):
    """P[s, q, q_next] exposes the shared virtual bit as a physical spin."""
    P = yastn.Tensor(config=config, s=(1, 1, -1), n=0)
    for q in (0, 1):
        values = np.zeros((2, 1, 1))
        values[q, 0, 0] = 1
        P.set_block(ts=(0, q, q), Ds=(2, 1, 1), val=values)
    return P


P_z2 = physical_copy(z2_config)
print("Copy tensor shape (s, q, q_next):", P_z2.get_shape())


## 3. From the edge-spin network to a square-lattice PEPS

- A standard PEPS associates a tensor with each site.
  Its virtual legs connect neighboring sites; its open physical leg labels the local spin.
  In the following, we convert the edge-spin network to the standard PEPS geometry.

- Split each $Q$ into four three-leg factors, two $R_1$ and two $R_2$.
  Both $R_1$ and $R_2$ enforce even parity:

  $$
  (R_1)_{q_0q_1q_2}=(R_2)_{q_0q_1q_2}
  =\delta_{(q_0+q_1+q_2)\bmod2,\,0}.
  $$

- Their entries are identical, but their signatures differ:
  `R1` has `(+,+,-)` and `R2` has `(-,-,+)` in `(q0,q1,q2)` order.

  ```text
                                (0)                            (0)
                                |                              ^
                                v                              |
                      (1) ---> [R1] ---> (2)          (1) <--- [R2] <--- (2)
  ```

- Place $R_1$ at the upper/right parts of each original vertex and $R_2$ at the lower/left parts.
  Every bond joins opposite signatures; the code below supplies the factor axes.

  $$
  Q_{uldr}=\frac12\sum_{i,j,k,m}
  (R_1)_{uij}(R_1)_{jkr}(R_2)_{dkm}(R_2)_{mil}.
  $$

  <p align="center">
    <img src="figures/toric_code_embedded_R1_R2.png" alt="R1 and R2 factors embedded in the original lattice with directed bonds; outlined P-R1-R2 groups become the A and B sites of a square-lattice PEPS" width="850">
  </p>

- We then group one $P$ with two neighboring $R$ factors, as illustrated in the figure.
  The blue groups become $A$ and the green groups become $B$.

**Exercise 3a:** contract the four supplied factors in the clockwise order
listed in the next cell. Sum the internal indices, divide by two, and keep
`(u,l,d,r)` as the output order. The result must equal `Q_z2`, including signatures.


In [ ]:
def splitter_tensors(config):
    """R1 has signatures (+,+,-); R2 has (-,-,+). Both have total charge zero."""
    leg = yastn.Leg(config, s=1, t=(0, 1), D=(1, 1))
    R1 = yastn.ones(config=config, legs=(leg, leg, leg.conj()), n=0)
    R2 = yastn.ones(config=config, legs=(leg.conj(), leg.conj(), leg), n=0)
    return R1, R2


R1, R2 = splitter_tensors(z2_config)
# Clockwise from the top: R1[u,i,j], R1[j,k,r], R2[d,k,m], R2[m,i,l].
# Internal labels i,j,k,m are summed; output legs are (u,l,d,r).
# TODO: Contract the two R1 and two R2 factors into Q[u,l,d,r].
raise NotImplementedError("Exercise 3a: contract the four R factors.")
assert Q_from_R.s == Q_z2.s
np.testing.assert_array_equal(Q_from_R.to_numpy(), Q_z2.to_numpy())
print("Verified: the R1/R2 contraction equals Q.")


### 3.1. Build the site tensors

- Contract the three tensors within each outlined group:

  $$
  \begin{aligned}
  A^s_{uldr}&=\sum_{x,y}(R_1)_{ulx}\,P^s_{xy}\,(R_2)_{dry},\\
  B^s_{uldr}&=\sum_{x,y}(R_2)_{xru}\,P^s_{xy}\,(R_1)_{yld}.
  \end{aligned}
  $$

- In code, the physical leg is **last**: `(u,l,d,r,s)`, with signatures `(+,+,-,-,+)`.
  The virtual bond dimension is $D=2$ and the physical spin dimension is $d=2$.
  The omitted factors of $1/2$ only change the state's overall normalization.

- Rotate the drawing clockwise by $45^\circ$ to obtain the PEPS lattice.
  Each new site represents one original edge spin: $A$ comes from horizontal edges and $B$ from vertical edges.
  Use the repeating unit cell

  ```text
                                                A B
                                                B A
  ```

- YASTN coordinates are `(row,column)`, with rows increasing downward.
  Use $A$ for even `row+column` and $B$ for odd `row+column`.
  Virtual arrows point right and down; the physical arrow points into each site.

**Exercise 3b:** implement the two contractions above with `yastn.einsum`.
Return both tensors in `(u,l,d,r,s)` order. The setup and signature checks are provided.


In [ ]:
def make_site_tensor(P, R1, R2, orientation):
    """Contract P, R1, R2 into legs (u,l,d,r,s), with signatures (+,+,-,-,+)."""
    if orientation == "A":
        # TODO: Contract R1[u,l,x], P[s,x,y], and R2[d,r,y].
        raise NotImplementedError("Exercise 3b: contract orientation A.")
    # TODO: Contract R2[x,r,u], P[s,x,y], and R1[y,l,d].
    raise NotImplementedError("Exercise 3b: contract orientation B.")


A_z2 = make_site_tensor(P_z2, R1, R2, "A")
B_z2 = make_site_tensor(P_z2, R1, R2, "B")
assert A_z2.s == B_z2.s == (1, 1, -1, -1, 1)
unit_cell = {
    (0, 0): A_z2, (0, 1): B_z2,
    (1, 0): B_z2, (1, 1): A_z2,
}
for site, tensor in unit_cell.items():
    print(f"{site}: shape {tensor.get_shape()}")


## 4. Converge the environment with YASTN

- To measure an observable, contract the infinite network $\langle\psi|O|\psi\rangle$
  and divide by $\langle\psi|\psi\rangle$.
  The bra and ket form a **double layer**; physical indices outside $O$ are summed.

- We use CTMRG to approximate the surrounding network with four corner tensors
  and four sets of edge tensors. Their bond dimension $\chi$ controls the approximation.

- The full CTMRG setup using YASTN is provided. Run the next cell and check that `Converged: True`.
  Reference: [YASTN CTM environment](https://yastn.github.io/yastn/fpeps/environment_ctm.html).


In [ ]:
# Provided: the complete CTMRG setup and convergence.
import yastn.tn.fpeps as fpeps

geometry = fpeps.SquareLattice(dims=(2, 2), boundary="infinite")
psi_infinite = fpeps.Peps(geometry, tensors=unit_cell)
env_ctm = fpeps.EnvCTM(psi_infinite, init="eye")

chi = 8
ctm_info = env_ctm.ctmrg_(
    opts_svd={"D_total": chi, "tol": 1e-12},
    max_sweeps=40,
    corner_tol=1e-10,
)
print(f"Converged: {ctm_info.converged}; sweeps: {ctm_info.sweeps}; "
      f"corner residual: {ctm_info.max_dsv:.2e}; boundary dimension: {ctm_info.max_D}")
assert ctm_info.converged


### 4.1. Measure stabilizers and magnetization

- After regrouping, each original star or plaquette operator acts on the four
  corners of a plaquette of the new PEPS lattice:

  ```text
       A(0,0) ---> B(0,1) ---> A(0,2)
          |          |          |
          v  ZZZZ    v  XXXX    v
          |          |          |
       B(1,0) ---> A(1,1) ---> B(1,2)
  ```

  The left plaquette measures $A_v$; the right measures $B_p$.

- Check the following for our constructed PEPS:

    - $\langle A_v\rangle=\langle B_p\rangle=1$.
    - Single-spin magnetizations $\langle X\rangle=\langle Z\rangle=0$ on both site types.

- The vanishing magnetizations show no single-spin magnetic order.
  Section 5 examines the state's entanglement.


In [ ]:
# Provided: operators, CTM measurements, and checks.
def spin_operator(matrix):
    """A one-qubit operator on the neutral physical leg."""
    operator = yastn.Tensor(config=z2_config, s=(1, -1), n=0)
    operator.set_block(ts=(0, 0), Ds=(2, 2), val=matrix)
    return operator


X = spin_operator(np.array([[0., 1.], [1., 0.]]))
Z = spin_operator(np.diag([1., -1.]))
av = env_ctm.measure_2x2(Z, Z, Z, Z, sites=((0, 0), (0, 1), (1, 0), (1, 1)))
bp = env_ctm.measure_2x2(X, X, X, X, sites=((0, 1), (0, 2), (1, 1), (1, 2)))
print(f"<A_v> = {av:.12f}; <B_p> = {bp:.12f}")
np.testing.assert_allclose([av, bp], [1, 1], atol=1e-10)

magnetizations = {}
for site in geometry.sites():
    for label, operator in [("X", X), ("Z", Z)]:
        value = env_ctm.measure_1site(operator, site=site)
        magnetizations[site, label] = value
        print(f"<{label}> at {tuple(site)} = {value:.3e}")
np.testing.assert_allclose(list(magnetizations.values()), 0, atol=1e-10)


## 5. Rényi entropy from a contractible patch

- Choose a region $R$ and trace out all spins outside it:
  $\rho_R=\operatorname{Tr}_{\bar R}|\psi\rangle\langle\psi|$ for a normalized state.
  This **reduced density matrix (RDM)** describes measurements within $R$.
  Its purity determines the second Rényi entropy:

  $$
  S_2(R)=-\log\operatorname{Tr}(\rho_R^2).
  $$

- For a large disk with one boundary, the entropy obeys the area law with a topological correction:

  $$
  S_2(R)=\alpha_2 L-\gamma+\cdots,\qquad \gamma=\log\mathcal D,
  $$

  where $L$ measures the boundary size and $\mathcal D$ is the **total anyon quantum dimension**.
  The toric code has four anyon types, each of quantum dimension one.
  Thus $\mathcal D=\sqrt{1+1+1+1}=2$ and we expect $\gamma=\log2$.


### 5.1. Construct the patch RDM $\rho_R$

- Take a rectangular patch with two rows and $w$ columns.
  The converged `env_ctm` approximates the contraction outside it, tracing out the surrounding spins.

- At each site of the patch, form the double layer $E_A=A\otimes A^*$ (or $E_B=B\otimes B^*$).
  Keep the physical ket and bra legs open, and fuse each virtual ket/bra pair.
  Conjugation reverses the bra arrows; a fused virtual leg follows the ket direction.
  For $w=2$, the unnormalized RDM $\widetilde\rho_R$ has the network

  ```text
         tl ---> t0 ---> t1 ---> tr
          ^       |       |       |
          |       v       v       v
         l0 ---> EA ---> EB ---> r0
          ^       |       |       |
          |       v       v       v
         l1 ---> EB ---> EA ---> r1
          ^       |       |       |
          |       v       v       v
         bl <--- b0 <--- b1 <--- br
  ```

  Here `EA` and `EB` are the double layers of $A$ and $B$; their physical legs are omitted.
  `t,b,l,r` denote top, bottom, left, and right CTM edges; `tl,tr,bl,br` are corners.

Contract this network in four steps:

1. **Build the side boundaries.** Contract `tl-l0-l1-bl` into `left` and
   `tr-r0-r1-br` into `right`. Their four remaining legs connect to the top CTM
   edge, upper site, lower site, and bottom CTM edge. Combine these four legs
   into one boundary index.

2. **Build each column from top to bottom.** Contract `t0` with the upper site,
   then the lower site, then `b0`; repeat for the other columns.
   Combine the four left virtual legs into `i` and the four right legs into `j`.
   The result is `column[i, j, p, q, P, Q]`: `p,q` are the top/bottom ket spins,
   and `P,Q` are their bra partners. These capital letters label physical indices,
   not the tensors $P$ and $Q$ from the construction.

   <p align="center">
     <img src="figures/ctm_strip_column.png" alt="Contract the top CTM edge, upper double layer, lower double layer, and bottom CTM edge; fuse the side bonds into a column with ket legs p and q above and bra legs P and Q below" width="850">
   </p>

   Fuse `i=(a,b,l,m)` and `j=(c,e,r,s)`. 

3. **Grow the patch from left to right.** Start with `left` and attach `column[0]`.
   Attach the remaining columns one at a time through the shared virtual index:

   ```text
                p q             p q
                | |             | |
   left --> [column[0]] --> [column[1]] --> right
                | |             | |
                P Q             P Q
   ```

   Keep every physical leg open. After each addition, group all accumulated ket
   indices into one index and all bra indices into another.

4. **Close and normalize.** Contract the remaining virtual index with `right`.
   Only the physical ket and bra indices remain, giving a matrix
   $\widetilde\rho_R$. Normalize it as
   $\rho_R=\widetilde\rho_R/\operatorname{Tr}\widetilde\rho_R$.

- The spin order is **column by column**, upper spin then lower spin.
  For $w=2$, the four-spin RDM is a $16\times16$ matrix.
  Its dimension grows as $2^{2w}\times2^{2w}$.

The first code cell below provides `strip_transfer`; read the function to understand
what it computes and run it unchanged.

**Exercise 5a:** in the second code cell below, complete the first-column contraction,
the column update, and the final closure and normalization in `ctm_strip_rdm`.
Index fusion and checks are provided.


In [ ]:
assert ctm_info.converged

# Provided: CTM boundaries, double layers, and column bookkeeping.
def strip_transfer(psi, env, width):
    """CTM boundaries and double-layer columns for a 2-by-width site patch."""
    # Side boundaries: arrows indicate signatures; (i) is the leg position.
    #
    # [tl]--a(1)-->             (0)a-->[tr]
    #   ^ x(0)                          | x(1)
    #   |                               |
    #   | x(2)                          v x(0)
    #  [l]--l(1)-->             (1)r-->[r]
    #   ^ y(0)                          | y(2)
    #   |                               |
    #   | y(2)                          v y(0)
    #  [l]--m(1)-->             (1)s-->[r]
    #   ^ z(0)                          | z(2)
    #   |                               |
    #   | z(1)                          v z(0)
    # [bl]<--b(0)--             (1)b<--[br]
    #
    # left[a(0), b(1), l(2), m(3)]; right[a(0), b(1), r(2), s(3)].
    top, bottom = env[(0, 0)], env[(1, 0)]
    left = yastn.einsum("xa,ylx,zmy,bz->ablm", top.tl, top.l, bottom.l, bottom.bl)
    top, bottom = env[(0, width - 1)], env[(1, width - 1)]
    right = yastn.einsum("ax,xry,ysz,zb->abrs", top.tr, top.r, bottom.r, bottom.br)
    columns = []
    for x in range(width):
        layers = []
        for y in (0, 1):
            A = psi[(y, x)]
            # E = A[p] x conj(A)[q]; keep both physical indices open.
            E = yastn.einsum("uldrp,ULDRq->uUlLdDrRpq", A, A.conj())
            layers.append(E.fuse_legs(axes=((0, 1), (2, 3), (4, 5), (6, 7), 8, 9), mode="hard"))
        # One column: [E] is a double layer after virtual ket/bra fusion.
        #
        # (0)a --> [t] --> c(2)
        #           | u(1)
        #           |
        #           v u(0)
        # (1)l --> [E] --> r(3)
        #           | v(2)
        #           |
        #           v v(0)
        # (1)m --> [E] --> s(3)
        #           | d(2)
        #           |
        #           v d(1)
        # (2)b <-- [b] <-- e(0)
        #
        #
        # Contract u, then v, then d. All physical legs stay open.
        column = yastn.einsum("auc,edb,ulvrpP,vmdsqQ->ablmcerspqPQ",
                              env[(0, x)].t, env[(1, x)].b, *layers, order="uvd")
        # Fuse (a,b,l,m) -> i and (c,e,r,s) -> j.
        # Result: column[i(0), j(1), p(2), q(3), P(4), Q(5)].
        columns.append(column.fuse_legs(axes=((0, 1, 2, 3), (4, 5, 6, 7), 8, 9, 10, 11), mode="hard"))
    # Fused directions: [left] --> i(0); j(0) --> [right].
    left = left.fuse_legs(axes=((0, 1, 2, 3),), mode="hard")
    right = right.fuse_legs(axes=((0, 1, 2, 3),), mode="hard")
    return left, columns, right


In [ ]:
def ctm_strip_rdm(psi, env, width):
    """Normalized RDM; spin order is top/bottom in each successive column."""
    left, columns, right = strip_transfer(psi, env, width)
    # 1. Attach the first column; a joins left(0) to column(0).
    #
    #                  p(2), q(3)
    #                      |
    #                      |
    # [left] ----a----> [column] ----b(1)---->
    #                      |
    #                      |
    #                  P(4), Q(5)
    #
    # TODO: Attach the first column to the left boundary.
    raise NotImplementedError("Exercise 5a: attach the first column.")
    # Fuse physical legs: rho[b(0), (p,q)(1), (P,Q)(2)].
    rho = rho.fuse_legs(axes=(0, (1, 2), (3, 4)), mode="hard")
    for column in columns[1:]:
        # 2. Append a column; a joins rho(0) to column(0).
        #
        #   i(1)              p(2), q(3)
        #     |                   |
        #     |                   |
        #   [rho] ----a----> [column] ----b(1)---->
        #     |                   |
        #     |                   |
        #   j(2)              P(4), Q(5)
        #
        # Keep ket legs together, then bra legs: (b,i,p,q,j,P,Q).
        # TODO: Attach the next column, retaining all physical legs.
        raise NotImplementedError("Exercise 5a: append a column.")
        # Fuse to rho[b(0), (i,p,q)(1), (j,P,Q)(2)].
        rho = rho.fuse_legs(axes=(0, (1, 2, 3), (4, 5, 6)), mode="hard")
    # 3. Close the boundary; a joins rho(0) to right(0).
    #
    #   i(1)
    #     |
    #     |
    #   [rho] ----a----> [right]
    #     |
    #     |
    #   j(2)
    #
    # Result: rho[i(0), j(1)]. Divide by its trace.
    # TODO: Close the right boundary and return the normalized RDM.
    raise NotImplementedError("Exercise 5a: close and normalize the RDM.")


rho_2x2 = ctm_strip_rdm(psi_infinite, env_ctm, 2)
print("2x2 patch RDM shape:", rho_2x2.get_shape())

rho_dense = rho_2x2.to_numpy()
np.testing.assert_allclose(rho_dense, rho_dense.conj().T, atol=1e-12)
np.testing.assert_allclose(np.trace(rho_dense), 1, atol=1e-12)
print("RDM checks passed: Hermitian and trace one.")


In [ ]:
def ctm_strip_purity(psi, env, width):
    """Contract two replicas to obtain Tr(rho**2) without storing the RDM."""
    left, columns, right = strip_transfer(psi, env, width)
    norm_boundary = left

    # Start with two left boundaries; fuse their open legs (a,c) into leg (0).
    purity_boundary = yastn.einsum("a,b->ab", left, left)
    purity_boundary = purity_boundary.fuse_legs(axes=((0, 1),), mode="hard")

    for column in columns:
        # One-copy norm: trace the physical legs (P=p, Q=q).
        # [norm_boundary] --a(0)--> (0)a--[tr_phys(column)]--b(1)-->
        norm_boundary = yastn.einsum("a,abpqpq->b", norm_boundary, column)
        # Two copies: each bra connects to the other copy's ket.
        #
        #             +----------- p,q -----------+
        #             |                           |
        # (0)a --> [copy 1] --> b(1)               |
        #             |                           |
        #            P,Q                          |
        #             |                           |
        # (0)c --> [copy 2] --> d(1)               |
        #             |                           |
        #             +---------------------------+
        #
        # P,Q: copy 1 legs (4,5) -> copy 2 legs (2,3); p,q go the other way.
        # Result: purity_transfer[a(0), c(1), b(2), d(3)].
        # TODO: Compute purity_transfer by contracting the bra and ket legs between two copies.
        raise NotImplementedError("Exercise 5b: construct the two-copy column.")
        # Fuse (a,c) and (b,d) into purity_transfer[left(0), right(1)].
        purity_transfer = purity_transfer.fuse_legs(axes=((0, 1), (2, 3)), mode="hard")
        # [purity_boundary] --(a,c)--> [purity_transfer] --(b,d)-->
        # Contract purity_boundary(0) with purity_transfer(0); the new boundary is purity_transfer(1).
        purity_boundary = yastn.einsum("a,ab->b", purity_boundary, purity_transfer)

    # Close the one-copy and two-copy boundaries.
    # [norm_boundary]   ----a----> [right]
    # [purity_boundary] --(a,c)--> [right x right]
    right_pair = yastn.einsum("a,b->ab", right, right)
    right_pair = right_pair.fuse_legs(axes=((0, 1),), mode="hard")
    norm = yastn.einsum("a,a->", norm_boundary, right).to_number()
    purity_numerator = yastn.einsum("a,a->", purity_boundary, right_pair).to_number()

    # Normalize by the squared one-copy norm.
    purity = purity_numerator / norm**2
    return purity


# Check the two-copy contraction against the explicit four-spin RDM.
purity_from_rdm = yastn.einsum("ij,ji->", rho_2x2, rho_2x2).to_number()
purity_from_transfer = ctm_strip_purity(psi_infinite, env_ctm, 2)
np.testing.assert_allclose(purity_from_transfer, purity_from_rdm, atol=1e-12)
print(f"2x2 purity: RDM = {purity_from_rdm:.8f}; transfer = {purity_from_transfer:.8f}")


### 5.3. Fit the topological correction

- A $2\times w$ site patch cuts $L=2(w+2)$ virtual bonds: $w$ at the top and bottom,
  and two at each side. Use widths $w=2,3,4,5,6$ and fit

  $$
  S_2=aL+b,\qquad \gamma=-b.
  $$

- For this exact toric-code PEPS, the strips already obey an exact lattice area law.
  For a generic state, both patch dimensions must be large compared with its correlation length.
  Extrapolate the fitted line to $L=0$ to read off its constant term.

- Run the provided scan. The plot uses $S_2/\log2$, so the expected intercept is $-1$
  and the extracted $\gamma/\log2$ is $1$.


In [ ]:
# Provided: the width scan, entropy fit, and plot.
def plot_renyi_scaling(boundaries, entropies):
    """Fit S2 = a L - gamma and plot the extrapolation to zero."""
    slope, intercept = np.polyfit(boundaries, entropies, 1)
    x = np.linspace(0, max(boundaries), 100)
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    ax.plot(x, (slope * x + intercept) / np.log(2), "--", label="Linear fit")
    ax.scatter(boundaries, np.asarray(entropies) / np.log(2), label="CTM")
    ax.scatter([0], [intercept / np.log(2)], color="black")
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.set(xlabel=r"Cut virtual bonds $L$", ylabel=r"$S_2/\log2$",
           title="Contractible patch entropy", xticks=[0, *boundaries])
    ax.legend()
    plt.show()
    return -intercept


widths = np.arange(2, 7)  # 2 through 6, spacing 1.
patch_boundaries = 2 * (widths + 2)
purities = [ctm_strip_purity(psi_infinite, env_ctm, int(w)) for w in widths]
patch_s2 = -np.log(purities)
print("Width   Cut virtual bonds   S2/log(2)")
for w, boundary, s2 in zip(widths, patch_boundaries, patch_s2):
    print(f"{w:5d}   {boundary:17d}   {s2 / np.log(2):.6f}")

gamma_disk = plot_renyi_scaling(patch_boundaries, patch_s2)
np.testing.assert_allclose(gamma_disk, np.log(2), atol=1e-12)
print(f"gamma = {gamma_disk:.9f} = log(2)")


## References

- [YASTN: symmetric tensor construction](https://yastn.github.io/yastn/tensor/init.html).
- [Schuch, Cirac, and Pérez-García (2010)](https://arxiv.org/abs/1001.3807):
  toric-code PEPS, virtual symmetries, and renormalization.
- [Kitaev (2003), Section 1](https://arxiv.org/abs/quant-ph/9707021):
  toric-code Hamiltonian, loop states, and fourfold torus degeneracy.
- [Schuch et al. (2012), Section II.E](https://arxiv.org/abs/1203.4816):
  toric code on a checkerboard square lattice.
- [Flammia et al. (2009)](https://arxiv.org/abs/0909.3305):
  topological corrections to Rényi entropies.
